# Building GPT with Hardware-Adaptive Optimizations

This notebook implements a GPT language model that **automatically optimizes for your hardware**, whether you're on NVIDIA datacenter GPUs (A100, H100), Apple Silicon (M1/M2/M3/M4), or CPU.

## What You'll Learn

1. **Character-level language modeling** with transformer architecture
2. **Multi-head attention** and how it enables learning long-range dependencies
3. **Hardware-specific optimizations** for maximum performance
4. **Mixed precision training** (BF16 on A100, FP16 on Apple Silicon)
5. **Memory-efficient training** strategies

## Hardware Optimizations

This notebook automatically detects your hardware and applies optimal settings:

**NVIDIA GPUs (A100, H100, etc.):**
- Mixed precision training (BF16) for 2-3x speedup
- Flash Attention 2 for 2-4x faster attention (if available)
- torch.compile for additional 1.5-2x speedup
- Fused AdamW optimizer
- Larger batch sizes (256) to utilize high memory
- TF32 acceleration on Ampere+ GPUs

**Apple Silicon (M1/M2/M3/M4):**
- MPS (Metal) backend for GPU acceleration
- Mixed precision (FP16) for 1.5-2x speedup
- Memory-efficient batched attention
- Adaptive batch sizing based on available memory
- Gradient accumulation for effective larger batches
- Unified memory optimizations

**CPU Fallback:**
- Smaller batch sizes
- Full FP32 precision
- Memory-efficient operations

## Configuration

All hyperparameters in one place. These will be auto-adjusted based on detected hardware.

In [ ]:
CONFIG = {
    # Reproducibility
    'seed': 1337,
    
    # Data (will be adjusted based on hardware)
    'batch_size': 64,  # Default, will be optimized per-device
    'block_size': 256,  # Context length
    
    # Model architecture
    'n_embed': 384,  # Embedding dimension
    'n_layers': 6,  # Number of transformer blocks
    'n_heads': 6,  # Number of attention heads
    'dropout': 0.2,  # Dropout rate
    
    # Training
    'learning_rate': 3e-4,
    'max_steps': 5000,
    'eval_interval': 100,
    'eval_iters': 200,
    
    # Hardware-specific (auto-configured)
    'use_mixed_precision': True,  # Enable mixed precision if supported
    'use_flash_attention': True,  # Use Flash Attention 2 if available (CUDA only)
    'use_compile': True,  # Use torch.compile if supported
    'use_fused_optimizer': True,  # Use fused AdamW (CUDA only)
    'use_gradient_accumulation': False,  # Enable for low-memory devices
    'gradient_accumulation_steps': 4,  # Steps to accumulate gradients
}

## Hardware Detection and Auto-Configuration

Detect the available hardware and automatically configure optimal settings.

In [ ]:
from aiml_notebooks import set_seed, detect_hardware

set_seed(CONFIG['seed'])

# Detect hardware and configure optimal settings
hw_config = detect_hardware(base_batch_size=CONFIG['batch_size'])

# Update CONFIG with hardware-specific settings
CONFIG['batch_size'] = hw_config.batch_size
CONFIG['use_mixed_precision'] = hw_config.precision != '32-true'
CONFIG['use_flash_attention'] = hw_config.use_flash_attention
CONFIG['use_compile'] = hw_config.use_compile
CONFIG['use_fused_optimizer'] = hw_config.use_fused_optimizer
if hw_config.gradient_accumulation_steps > 1:
    CONFIG['use_gradient_accumulation'] = True
    CONFIG['gradient_accumulation_steps'] = hw_config.gradient_accumulation_steps

# Store hardware config for later use
DEVICE_TYPE = hw_config.device_type
device = hw_config.device
precision_type = hw_config.precision
pin_memory = hw_config.pin_memory

## Load Tiny Shakespeare Dataset

We'll train on Shakespeare's works - a classic character-level language modeling task.

In [ ]:
import requests

response = requests.get("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt")
text = response.text

print(f"Dataset length: {len(text):,} characters")
print(f"\nFirst 200 characters:")
print(text[:200])

## Build Character-Level Tokenizer

Create a simple vocabulary mapping each unique character to an integer.

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars[:20])}...")
print(f"\nTest encoding:")
test_text = "Hello"
encoded = encode(test_text)
print(f"  '{test_text}' → {encoded}")
print(f"  {encoded} → '{decode(encoded)}'")

## Create Train/Validation Split

Split into 90% training and 10% validation sets.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")
print(f"Train/val split: {len(train_data)/len(data)*100:.1f}% / {len(val_data)/len(data)*100:.1f}%")

## Lightning DataModule

Create a PyTorch Lightning DataModule with hardware-optimized data loading settings.

**DataLoader Settings:**
- **num_workers=0**: In-memory data doesn't need parallel loading
- **pin_memory**: True for CUDA (faster CPU→GPU transfers), False for unified memory (MPS/CPU)
- **persistent_workers**: Not needed when num_workers=0

In [ ]:
import lightning as L
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    """Character-level dataset that returns random sequences."""
    
    def __init__(self, data, block_size, num_samples):
        self.data = data
        self.block_size = block_size
        self.num_samples = num_samples
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        i = torch.randint(len(self.data) - self.block_size, (1,)).item()
        x = self.data[i:i+self.block_size]
        y = self.data[i+1:i+self.block_size+1]
        return x, y

class ShakespeareDataModule(L.LightningDataModule):
    """DataModule for Shakespeare character-level data."""
    
    def __init__(self, train_data, val_data, batch_size, block_size, eval_iters, pin_memory):
        super().__init__()
        self.train_data = train_data
        self.val_data = val_data
        self.batch_size = batch_size
        self.block_size = block_size
        self.eval_iters = eval_iters
        self.pin_memory = pin_memory
    
    def train_dataloader(self):
        dataset = CharDataset(self.train_data, self.block_size, num_samples=100000)
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            num_workers=0,
            pin_memory=self.pin_memory,
            persistent_workers=False
        )
    
    def val_dataloader(self):
        dataset = CharDataset(self.val_data, self.block_size, 
                            num_samples=self.eval_iters * self.batch_size)
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            num_workers=0,
            pin_memory=self.pin_memory,
            persistent_workers=False
        )

datamodule = ShakespeareDataModule(
    train_data=train_data,
    val_data=val_data,
    batch_size=CONFIG['batch_size'],
    block_size=CONFIG['block_size'],
    eval_iters=CONFIG['eval_iters'],
    pin_memory=pin_memory
)

print(f"DataModule created")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Block size: {CONFIG['block_size']}")
print(f"  Pin memory: {pin_memory}")
print(f"  Tokens per batch: {CONFIG['batch_size'] * CONFIG['block_size']:,}")

## Check Flash Attention Availability

Flash Attention 2 provides 2-4x speedup on CUDA GPUs. If not installed, we'll use optimized standard attention.

In [ ]:
from aiml_notebooks import check_flash_attention

FLASH_AVAILABLE = check_flash_attention()

if CONFIG['use_flash_attention'] and DEVICE_TYPE == 'cuda':
    if FLASH_AVAILABLE:
        print("✓ Flash Attention 2 is available")
    else:
        print("ℹ Flash Attention 2 not available")
        print("  Install with: pip install flash-attn")
        print("  Will use optimized standard attention instead")
        CONFIG['use_flash_attention'] = False
else:
    print(f"Flash Attention: Not applicable for {DEVICE_TYPE}")

## Multi-Head Attention Layer

Implement attention with two backends:
1. **Flash Attention**: Used on CUDA if available (2-4x faster, less memory)
2. **Optimized Standard**: Batched attention for all other platforms

In [ ]:
import torch.nn as nn
from torch.nn import functional as F
from aiml_notebooks import flash_attention_func

class MultiHeadAttention(nn.Module):
    """Multi-head attention with hardware-optimized backends."""
    
    def __init__(self, n_embed, n_heads, block_size, dropout, use_flash=False):
        super().__init__()
        assert n_embed % n_heads == 0
        
        self.n_heads = n_heads
        self.head_size = n_embed // n_heads
        self.n_embed = n_embed
        self.use_flash = use_flash and FLASH_AVAILABLE
        
        # Combined QKV projection (more efficient)
        self.qkv = nn.Linear(n_embed, 3 * n_embed, bias=False)
        self.proj = nn.Linear(n_embed, n_embed)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.proj_dropout = nn.Dropout(dropout)
        
        # Causal mask (only for standard attention)
        if not self.use_flash:
            self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    
    def forward(self, x):
        B, T, C = x.shape
        
        # Combined QKV projection
        qkv = self.qkv(x)  # (B, T, 3*n_embed)
        q, k, v = qkv.chunk(3, dim=-1)  # Each is (B, T, n_embed)
        
        # Reshape to (B, T, n_heads, head_size)
        q = q.view(B, T, self.n_heads, self.head_size)
        k = k.view(B, T, self.n_heads, self.head_size)
        v = v.view(B, T, self.n_heads, self.head_size)
        
        if self.use_flash:
            # Use unified flash attention interface (auto-detects Flash Attention 2)
            out = flash_attention_func(
                q, k, v,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                causal=True
            )  # Returns (B, T, n_heads, head_size)
            out = out.contiguous().view(B, T, self.n_embed)
        else:
            # Standard scaled dot-product attention (batched)
            q = q.transpose(1, 2)  # (B, n_heads, T, head_size)
            k = k.transpose(1, 2)
            v = v.transpose(1, 2)
            
            att = (q @ k.transpose(-2, -1)) * (self.head_size ** -0.5)
            att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            out = att @ v  # (B, n_heads, T, head_size)
            out = out.transpose(1, 2).contiguous().view(B, T, self.n_embed)
        
        # Output projection
        out = self.proj_dropout(self.proj(out))
        return out

print(f"Multi-head attention configured")
print(f"  Backend: {'Flash Attention 2 (with fallback)' if FLASH_AVAILABLE and CONFIG['use_flash_attention'] else 'Optimized standard'}")

## Feed-Forward Network

Standard position-wise feed-forward network with GELU activation.

In [ ]:
class FeedForward(nn.Module):
    """Feed-forward network with GELU activation."""
    
    def __init__(self, n_embed, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.GELU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)

## Transformer Block

A complete transformer decoder block: attention → add & norm → feed-forward → add & norm.

In [ ]:
class Block(nn.Module):
    """Transformer decoder block."""
    
    def __init__(self, n_embed, n_heads, block_size, dropout, use_flash=False):
        super().__init__()
        self.sa = MultiHeadAttention(n_embed, n_heads, block_size, dropout, use_flash)
        self.ffwd = FeedForward(n_embed, dropout)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
    
    def forward(self, x):
        x = x + self.sa(self.ln1(x))  # Attention with residual
        x = x + self.ffwd(self.ln2(x))  # Feed-forward with residual
        return x

## Complete GPT Model

Full GPT model with hardware-adaptive optimizations using PyTorch Lightning.

In [ ]:
import time

class GPTLanguageModel(L.LightningModule):
    """GPT with automatic hardware optimization."""
    
    def __init__(self, vocab_size, n_embed=CONFIG['n_embed'], 
                 n_layers=CONFIG['n_layers'], n_heads=CONFIG['n_heads'],
                 block_size=CONFIG['block_size'], dropout=CONFIG['dropout'],
                 learning_rate=CONFIG['learning_rate'],
                 use_flash=CONFIG['use_flash_attention']):
        super().__init__()
        self.save_hyperparameters()
        self.block_size = block_size
        
        # Model components
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.ModuleList([
            Block(n_embed, n_heads, block_size, dropout, use_flash) 
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)
        
        # Training time tracking
        self.train_start_time = None
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        # Embeddings
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        
        # Transformer blocks
        for block in self.blocks:
            x = block(x)
        
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            targets_flat = targets.view(B*T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        
        return logits, loss
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits, loss = self(x, y)
        
        # Handle gradient accumulation
        if CONFIG['use_gradient_accumulation']:
            loss = loss / CONFIG['gradient_accumulation_steps']
        
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=False)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits, loss = self(x, y)
        self.log('val_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss
    
    def on_train_start(self):
        self.train_start_time = time.time()
    
    def on_train_batch_end(self, outputs, batch, batch_idx):
        if self.train_start_time is not None:
            elapsed = time.time() - self.train_start_time
            self.log('train_time_seconds', elapsed, prog_bar=False)
            
            # Calculate throughput
            effective_batch = CONFIG['batch_size']
            if CONFIG['use_gradient_accumulation']:
                effective_batch *= CONFIG['gradient_accumulation_steps']
            tokens_processed = (batch_idx + 1) * effective_batch * CONFIG['block_size']
            throughput = tokens_processed / elapsed
            self.log('tokens_per_second', throughput, prog_bar=False)
    
    def on_train_end(self):
        if self.train_start_time is not None:
            total_time = time.time() - self.train_start_time
            print(f"\nTotal training time: {total_time:.2f}s ({total_time/60:.2f} min)")
    
    def configure_optimizers(self):
        # Use fused AdamW on CUDA if available
        if CONFIG['use_fused_optimizer'] and DEVICE_TYPE == 'cuda':
            return torch.optim.AdamW(
                self.parameters(), 
                lr=self.hparams.learning_rate,
                fused=True
            )
        else:
            return torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate)
    
    def generate(self, idx, max_new_tokens):
        """Generate new tokens autoregressively."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = GPTLanguageModel(vocab_size)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Model size (FP32): ~{total_params * 4 / 1e6:.1f} MB")
if CONFIG['use_mixed_precision']:
    print(f"  Model size (FP16/BF16): ~{total_params * 2 / 1e6:.1f} MB")

## Apply torch.compile (Optional)

JIT compilation for additional 1.5-2x speedup on compatible hardware (works best on CUDA).

In [ ]:
from aiml_notebooks import apply_torch_compile

if CONFIG['use_compile']:
    model = apply_torch_compile(model, mode='max-autotune')
else:
    print("torch.compile disabled for this hardware")

## Training Setup

Configure PyTorch Lightning Trainer with hardware-optimized settings.

In [ ]:
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint

logger = CSVLogger('logs', name=f'gpt_{DEVICE_TYPE}')

checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    filename='best-{epoch:02d}-{val_loss:.4f}'
)

# Gradient accumulation if enabled
accumulate_grad_batches = 1
if CONFIG['use_gradient_accumulation']:
    accumulate_grad_batches = CONFIG['gradient_accumulation_steps']

trainer = L.Trainer(
    max_steps=CONFIG['max_steps'],
    val_check_interval=CONFIG['eval_interval'],
    accelerator='auto',
    devices=1,
    precision=precision_type if CONFIG['use_mixed_precision'] else '32-true',
    logger=logger,
    callbacks=[checkpoint_callback],
    enable_progress_bar=True,
    log_every_n_steps=1,
    accumulate_grad_batches=accumulate_grad_batches,
    benchmark=(DEVICE_TYPE == 'cuda'),  # cuDNN benchmarking on CUDA
)

effective_batch = CONFIG['batch_size'] * accumulate_grad_batches
print(f"\nTrainer Configuration:")
print(f"  Max steps: {CONFIG['max_steps']}")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Effective batch: {effective_batch}")
print(f"  Precision: {precision_type if CONFIG['use_mixed_precision'] else '32-true'}")
print(f"  Tokens per step: {effective_batch * CONFIG['block_size']:,}")

## Train the Model

Run training with all hardware-specific optimizations enabled.

In [ ]:
print(f"Starting training on {DEVICE_TYPE.upper()}...\n")
trainer.fit(model, datamodule)

## Plot Training Curves and Performance Metrics

Visualize training progression, loss curves, and throughput.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics = pd.read_csv(f'{logger.log_dir}/metrics.csv')

train_metrics = metrics[['step', 'train_loss']].dropna()
val_metrics = metrics[['step', 'val_loss']].dropna()
time_metrics = metrics[['step', 'train_time_seconds']].dropna()
throughput_metrics = metrics[['step', 'tokens_per_second']].dropna()

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 10))

# Loss curves
ax1.plot(train_metrics['step'], train_metrics['train_loss'],
         label='Train', linewidth=2, alpha=0.7, color='#4ECDC4')
ax1.plot(val_metrics['step'], val_metrics['val_loss'],
         label='Validation', marker='o', linewidth=2, markersize=3, color='#FF6B6B')
ax1.set_xlabel('Step', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title(f'Training Progress ({DEVICE_TYPE.upper()})', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Training time
ax2.plot(time_metrics['step'], time_metrics['train_time_seconds'] / 60,
         linewidth=2, color='#95E1D3')
ax2.set_xlabel('Step', fontsize=12)
ax2.set_ylabel('Elapsed Time (minutes)', fontsize=12)
ax2.set_title('Training Time', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Throughput
ax3.plot(throughput_metrics['step'], throughput_metrics['tokens_per_second'],
         linewidth=2, color='#F38181')
ax3.set_xlabel('Step', fontsize=12)
ax3.set_ylabel('Tokens/Second', fontsize=12)
ax3.set_title('Training Throughput', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Steps per second
time_metrics['steps_per_sec'] = time_metrics['step'] / time_metrics['train_time_seconds']
ax4.plot(time_metrics['step'], time_metrics['steps_per_sec'],
         linewidth=2, color='#A8E6CF')
ax4.set_xlabel('Step', fontsize=12)
ax4.set_ylabel('Steps/Second', fontsize=12)
ax4.set_title('Training Speed', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Performance statistics
final_train_loss = train_metrics['train_loss'].iloc[-1]
final_val_loss = val_metrics['val_loss'].iloc[-1]
total_time = time_metrics['train_time_seconds'].iloc[-1]
avg_throughput = throughput_metrics['tokens_per_second'].mean()
avg_steps_per_sec = time_metrics['steps_per_sec'].mean()

print(f"\n{'='*70}")
print(f"TRAINING STATISTICS ({DEVICE_TYPE.upper()})")
print(f"{'='*70}")
print(f"\nPerformance:")
print(f"  Total steps: {len(train_metrics):,}")
print(f"  Total time: {total_time:.2f}s ({total_time/60:.2f} minutes)")
print(f"  Average speed: {avg_steps_per_sec:.2f} steps/sec")
print(f"  Average throughput: {avg_throughput:,.0f} tokens/sec")
print(f"  Total tokens: {len(train_metrics) * effective_batch * CONFIG['block_size']:,}")
print(f"\nModel Quality:")
print(f"  Final train loss: {final_train_loss:.4f}")
print(f"  Final val loss: {final_val_loss:.4f}")
print(f"\nOptimizations Enabled:")
print(f"  Device: {DEVICE_TYPE.upper()}")
print(f"  Mixed Precision: {'✓' if CONFIG['use_mixed_precision'] else '✗'} ({precision_type if CONFIG['use_mixed_precision'] else 'FP32'})")
print(f"  Flash Attention: {'✓' if FLASH_AVAILABLE and CONFIG['use_flash_attention'] else '✗'}")
print(f"  torch.compile: {'✓' if CONFIG['use_compile'] else '✗'}")
print(f"  Fused Optimizer: {'✓' if CONFIG['use_fused_optimizer'] else '✗'}")
print(f"  Gradient Accumulation: {'✓' if CONFIG['use_gradient_accumulation'] else '✗'}")
print(f"  Batch Size: {CONFIG['batch_size']}")
print(f"  Effective Batch: {effective_batch}")
print(f"{'='*70}")

## Generate Shakespeare-like Text

Use the trained model to generate new text in Shakespeare's style.

In [ ]:
from aiml_notebooks import get_device

device = get_device()

# Handle torch.compile wrapper
if hasattr(model, '_orig_mod'):
    print("Note: First generation may be slow due to torch.compile...")

model = model.to(device)
model.eval()

# Generate from empty context
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_text = decode(model.generate(context, max_new_tokens=500)[0].tolist())

print("\nGenerated text:")
print("="*80)
print(generated_text)
print("="*80)

## Key Takeaways

### Hardware-Adaptive Training

This notebook demonstrates how to write **hardware-agnostic** deep learning code that automatically optimizes for different platforms:

**NVIDIA GPUs (A100, H100):**
- **BF16 mixed precision**: 2-3x speedup with better stability than FP16
- **Flash Attention 2**: 2-4x faster attention with reduced memory
- **torch.compile**: Additional 1.5-2x speedup through JIT compilation
- **Fused optimizers**: Single-kernel parameter updates
- **Large batches**: Utilize 40-80GB dedicated memory
- **TF32**: Automatic acceleration on Ampere+ architectures
- **Combined speedup**: 5-8x faster than baseline

**Apple Silicon (M1/M2/M3/M4):**
- **MPS backend**: Native Metal GPU acceleration
- **FP16 mixed precision**: 1.5-2x speedup (BF16 not well-supported)
- **Adaptive batching**: Auto-adjusts based on unified memory (8-128GB)
- **Gradient accumulation**: Simulates larger batches on low-memory devices
- **Unified memory**: No CPU↔GPU transfers needed
- **Combined speedup**: 2-8x faster than CPU (depends on chip)

### Transformer Architecture Insights

**Multi-head attention** is the key innovation:
- Allows model to attend to different aspects of the context
- Each head learns different patterns (syntax, semantics, long-range dependencies)
- Parallel computation across heads enables efficient training
- Causal masking ensures autoregressive property

**Residual connections** enable deep networks:
- Gradient flow through addition operations
- Each block can learn refinements to existing representations
- Allows training 6+ layers without vanishing gradients

**Layer normalization** stabilizes training:
- Pre-norm architecture (norm before attention/FFN) is standard in modern transformers
- Prevents activation explosion in deep networks
- Enables higher learning rates

### Performance Optimization Strategy

**Optimize in this order:**
1. **Mixed precision** (biggest impact, easy to enable)
2. **Optimal batch size** (utilize available memory)
3. **Flash Attention** (if on CUDA, 2-4x speedup)
4. **torch.compile** (if PyTorch 2.0+, 1.5-2x speedup)
5. **Gradient accumulation** (if memory-constrained)
6. **Multi-GPU** (if available, linear scaling)

**When to use what:**
- **Prototyping**: Apple Silicon with MPS (convenient, fast enough)
- **Training large models**: NVIDIA datacenter GPUs (A100, H100)
- **Fine-tuning**: Either platform works well
- **Inference**: Quantization + any platform

### Further Exploration

Try these experiments:
1. **Increase model size**: More layers (8-12) or larger embeddings (512, 768)
2. **Longer context**: Increase block_size to 512 or 1024
3. **Better tokenization**: Use BPE (byte-pair encoding) instead of characters
4. **More data**: Train on larger text corpus
5. **Learning rate schedule**: Add warmup and decay
6. **Multi-GPU**: Use DDP for data parallelism

**Key lesson**: Write hardware-agnostic code that adapts to available resources. The same codebase runs optimally on datacenter GPUs, laptops, and workstations.